# Load data

In [1]:
import pandas as pd
import numpy as np
import time

# --- CONFIG ---
SHORT = False
# SHORT = True

if SHORT:
    file_path = 'data/engineered_short.parquet'
else:
    file_path = 'data/engineered.parquet'

print(f"Loading {file_path}...")
df = pd.read_parquet(file_path)

# Define Features and Target
target = 'meter_reading'
# Exclude timestamp (not a number) and location_id (redundant with building_id)
exclude = [target, 'timestamp', 'location_id']
features = [col for col in df.columns if col not in exclude]

# Ensure building_id is treated as a category (vital for LightGBM/XGBoost)
df['building_id'] = df['building_id'].astype('category')

print(f"Dataset Loaded: {df.shape[0]:,} rows")
print(f"Features: {features}")

Loading data/engineered.parquet...
Dataset Loaded: 17,664,462 rows
Features: ['building_id', 'temperature_f', 'apparent_temperature_f', 'CDH', 'HDH', 'temp_roll_3h', 'hour', 'day_week', 'month', 'is_weekend', 'sqft', 'primary_space_usage', 'sub_type', 'year_built', 'number_of_floors']


In [2]:
# This will show you exactly which columns have NaNs and how many
print("--- NaN Count Per Column ---")
print(df[features + [target]].isna().sum())
print(df.head())

--- NaN Count Per Column ---
building_id                      0
temperature_f                    0
apparent_temperature_f           0
CDH                              0
HDH                              0
temp_roll_3h                     0
hour                             0
day_week                         0
month                            0
is_weekend                       0
sqft                             0
primary_space_usage              0
sub_type                         0
year_built                 8415148
number_of_floors          12987030
meter_reading                    0
dtype: int64
            timestamp            building_id  meter_reading  location_id  \
0 2016-01-01 00:00:00  Robin_public_Carolina      36.438000           11   
1 2016-01-01 01:00:00  Robin_public_Carolina      70.750000           11   
2 2016-01-01 02:00:00  Robin_public_Carolina      74.311996           11   
3 2016-01-01 03:00:00  Robin_public_Carolina      73.438004           11   
4 2016-01-01 04:00

# Splits

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# --- 1. TARGET NORMALIZATION ---
# Create Energy Use Intensity (EUI) target
# We add a tiny epsilon (1e-5) to sqft just in case of zeros, though usually not needed
df['eui_target'] = df['meter_reading'] / df['sqft']

# --- 2. FEATURE SELECTION ---
# We now exclude the raw meter_reading and the new eui_target from features
target = 'eui_target'
raw_target = 'meter_reading'

to_exclude = [target, raw_target, 'timestamp', 'location_id', 'building_id', 'site_id',
              'is_weekend', 
              # 'CDH', 'HDH'
             ] 

features = [col for col in df.columns if col not in to_exclude]

# --- 3. TOGGLE SWITCH ---
SPLIT_STRATEGY = 'time' # Set to 'group' for your Generalist Demo

# 4. SPLIT LOGIC
if SPLIT_STRATEGY == 'time':
    print(f"Applying TIME-BASED split (Forecasting)...")
    time_split = '2016-10-01' if SHORT else '2017-08-01'
    train_mask = df['timestamp'] < time_split
    val_mask = df['timestamp'] >= time_split
    
    X_train, y_train = df.loc[train_mask, features], df.loc[train_mask, target]
    X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, target]
    
    # Keep raw values for final evaluation
    y_val_raw = df.loc[val_mask, raw_target]
    
elif SPLIT_STRATEGY == 'group':
    print(f"Applying GROUP-BASED split (New Building Generalization)...")
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['building_id']))
    
    X_train, y_train = df.iloc[train_idx][features], df.iloc[train_idx][target]
    X_val, y_val = df.iloc[val_idx][features], df.iloc[val_idx][target]
    
    # Keep raw values for final evaluation
    y_val_raw = df.iloc[val_idx][raw_target]

print(f"Done. Training on {len(features)} features.")
print(f"Target: {target}")

Applying TIME-BASED split (Forecasting)...
Done. Training on 13 features.
Target: eui_target


# LightGBM (Gradient Boosted Decision Tree)

# Save LightGBM Model

# XGBoost

# Linear Regression

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np
import time

# --- PREPROCESSING FOR LINEAR REGRESSION ---
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()

# 1. Convert categories to codes (as we did before)
for col in X_train_lr.select_dtypes(['category']).columns:
    X_train_lr[col] = X_train_lr[col].cat.codes
    X_val_lr[col] = X_val_lr[col].cat.codes

# 2. NEW: Handle NaNs (Linear Regression cannot handle missing values)
# We fill NaNs with the median of the training set
X_train_lr = X_train_lr.fillna(X_train_lr.median())
X_val_lr = X_val_lr.fillna(X_train_lr.median()) # Use train median for val to avoid leakage

# --- TRAIN ---
model_lr = LinearRegression()
print("Starting Linear Regression Training...")
start_train = time.time()

# This should now run without the ValueError
model_lr.fit(X_train_lr, y_train)

print(f"Training Duration: {time.time() - start_train:.2f}s")

# --- EVALUATION ---
preds_eui_lr = model_lr.predict(X_val_lr)
preds_raw_lr = preds_eui_lr * X_val_lr['sqft']

rmse_lr = np.sqrt(mean_squared_error(y_val_raw, preds_raw_lr))
mae_lr = mean_absolute_error(y_val_raw, preds_raw_lr)
cvrmse_lr = (rmse_lr / y_val_raw.mean()) * 100
nmae_lr = (mae_lr / y_val_raw.mean()) * 100

print("-" * 30)
print(f"Linear Regression Final RMSE: {rmse_lr:.4f}")
print(f"CVRMSE:                      {cvrmse_lr:.2f}%")
print(f"Final MAE:                   {mae_lr:.4f}")
print(f"NMAE:                        {nmae_lr:.2f}%")
print("-" * 30)

Starting Linear Regression Training...
Training Duration: 2.87s
------------------------------
Linear Regression Final RMSE: 217.4454
CVRMSE:                      147.03%
Final MAE:                   96.0936
NMAE:                        64.98%
------------------------------


In [5]:
# Save
import joblib

joblib.dump(model_lr, 'models/linear_regression.pkl')

print("Linear regression successfully saved")

Linear regression successfully saved


# Bayesian